# Картограммы и классификация данных

## Картограмма

Один из самых популярных способов картографического изображения — **картограмма** (в англоязычной литературе — choropleth или хороплет).
На такой карте территориальные единицы (районы, регионы или страны) закрашиваются разными оттенками в зависимости от значения показателя.

Важное ограничение: картограмма корректно работает **только с относительными показателями** — плотностью, долями, средними значениями. Абсолютные числа (например, общая численность населения района) использовать нельзя, потому что площадь территориальных единиц всегда разная. Большой по площади район с небольшим населением будет визуально доминировать на карте и казаться «значимым», хотя по сути может уступать компактному, но плотно заселённому соседу. Нормировка на площадь, численность или другой знаменатель убирает этот эффект и делает сравнение честным.

Именно поэтому в этом ноутбуке мы работаем с **плотностью населения** (чел/км²), а не с абсолютной численностью.

## Классификация

Когда мы строим картограмму, нам нужно разбить числовые данные на интервалы и назначить каждому интервалу цвет. Казалось бы, технический вопрос — но выбор метода классификации кардинально меняет то, как мы отображаем данные на карте.

**В этом ноутбуке:**

- разберём основные методы классификации из библиотеки `mapclassify`
- посмотрим, как одни и те же данные выглядят при разных методах

**Данные:**

- **spb_admin.gpkg** — полигональные данные о границах районов и округов Санкт-Петербурга.  
  Источник: материалы курса «Методы пространственного анализа», НИУ ВШЭ (Р. Гончаров). Можно скачать [тут](https://github.com/bella-mir/geoPython/blob/main/chapters/module_1/data/spb_admin.gpkg)


## 0. Импорт библиотек


In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import mapclassify as mc
import numpy as np


## 1. Подготовка данных


In [ ]:
# Читаем данные об округах Санкт-Петербурга
spb = gpd.read_file("../data/spb_admin.gpkg", layer="okrug")

# Переводим в метрическую проекцию, чтобы считать площадь в км²
spb = spb.to_crs(spb.estimate_utm_crs())

# Считаем площадь и плотность населения
spb["area_km2"] = spb.geometry.area / 1_000_000
spb["pop_density"] = spb["Popul"] / spb["area_km2"]

spb[["NAME", "Popul", "area_km2", "pop_density"]].head(10)

### Смотрим на распределение данных

Перед выбором метода классификации всегда смотрим на распределение: есть ли выбросы, как сгруппированы значения, симметрично ли распределение.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Гистограмма
spb["pop_density"].hist(bins=15, color="#8856a7", ax=ax1)
ax1.set_title("Распределение плотности населения")
ax1.set_xlabel("Плотность, чел/км²")
ax1.set_ylabel("Количество округов")

# Box plot — удобно видеть медиану и выбросы
spb["pop_density"].plot.box(ax=ax2, color="#8856a7")
ax2.set_title("Box plot")
ax2.set_ylabel("Плотность, чел/км²")

plt.tight_layout()
plt.show()

# Основные статистики
spb["pop_density"].describe().round(1)

### Карта без классификации

По умолчанию GeoPandas использует непрерывную шкалу. Посмотрим, как это выглядит.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
spb.plot(column="pop_density", cmap="BuPu", linewidth=0.8,
         edgecolor="0.75", legend=True, ax=ax)
ax.set_title("Плотность населения, чел/км²\n(непрерывная шкала)")
ax.axis("off")
plt.show()

## 2. Методы классификации

Разберём сначала на одном примере, как работает классификация изнутри, а потом посмотрим на все методы.


### Шаг 1. Создаём классификатор

Объект-классификатор принимает числовой столбец и возвращает два атрибута:

- `bins` — границы классов (включая максимальное значение)
- `yb` — номер класса для каждого объекта (0, 1, 2, …)


In [ ]:
classifier = mc.EqualInterval(spb["pop_density"], k=5)

print("Границы классов:", classifier.bins.round(1))
print("Класс каждого округа:", classifier.yb)
print("Количество объектов в каждом классе:", np.bincount(classifier.yb))


### Шаг 2. Гистограмма с границами классов

Нанесём границы из `classifier.bins` на гистограмму распределения — сразу видно, как классы ложатся на данные.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(spb["pop_density"], bins=20, color="#8856a7", edgecolor="white", alpha=0.8)

# classifier.bins[:-1] — все границы кроме последней (максимум нам не нужен)
for bound in classifier.bins[:-1]:
    ax.axvline(bound, color="black", linewidth=0.8, linestyle="--", alpha=0.5)

ax.set_xlabel("Плотность, чел/км²")
ax.set_ylabel("Количество округов")
ax.set_title("Equal Interval: распределение с границами классов")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


### Шаг 3. Карта с классификацией

Передаём метод в `scheme=` — GeoPandas сам строит классификатор внутри и назначает цвета.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

spb.plot(column="pop_density", scheme="EqualInterval", k=5,
         cmap="BuPu", linewidth=0.8, edgecolor="0.75", legend=True, ax=ax)

ax.set_title("Equal Interval — равные интервалы")
ax.axis("off")
leg = ax.get_legend()
if leg:
    leg.set_bbox_to_anchor((1.05, 1))
plt.tight_layout()
plt.show()


Эти три шага — классификатор, гистограмма с границами, карта — придётся повторять для каждого из пяти методов. Чтобы не писать один и тот же код шесть раз, оформим его в **функцию**.

Функция `plot_classification` принимает GeoDataFrame, столбец и название метода, а внутри выполняет ровно те шаги, которые мы только что разобрали — и показывает гистограмму и карту рядом.


In [ ]:
def plot_classification(gdf, column, scheme, k=5, bins=None, title=None):

    fig, (ax_hist, ax_map) = plt.subplots(1, 2, figsize=(14, 5))

    # Шаг 1: создаём классификатор по названию метода
    if scheme == "EqualInterval":
        classifier = mc.EqualInterval(gdf[column], k=k)
    elif scheme == "Quantiles":
        classifier = mc.Quantiles(gdf[column], k=k)
    elif scheme == "NaturalBreaks":
        classifier = mc.NaturalBreaks(gdf[column], k=k)
    elif scheme == "MaximumBreaks":
        classifier = mc.MaximumBreaks(gdf[column], k=k)
    elif scheme == "StdMean":
        classifier = mc.StdMean(gdf[column])   # StdMean не принимает k
    elif scheme == "UserDefined":
        classifier = mc.UserDefined(gdf[column], bins)

    # Шаг 2: гистограмма с окраской по классам
    _, bin_edges, patches = ax_hist.hist(gdf[column], bins=20, edgecolor="white", alpha=0.85)

    # Красим каждый столбец в цвет его класса (та же палитра BuPu, что на карте)
    n_classes = len(classifier.bins) - 1
    cmap = plt.cm.BuPu
    for patch, left in zip(patches, bin_edges[:-1]):
        class_idx = min(np.searchsorted(classifier.bins[1:], left), n_classes - 1)
        patch.set_facecolor(cmap((class_idx + 1.5) / (n_classes + 1)))

    # Шаг 3: вертикальные линии — границы классов
    for bound in classifier.bins[:-1]:
        ax_hist.axvline(bound, color="black", linewidth=0.8, linestyle="--", alpha=0.5)

    ax_hist.set_xlabel("Плотность, чел/км²")
    ax_hist.set_ylabel("Количество округов")
    ax_hist.set_title("Распределение с границами классов")
    ax_hist.spines[["top", "right"]].set_visible(False)

    # Шаг 4: карта с классификацией
    base = dict(column=column, cmap="BuPu", linewidth=0.8,
                edgecolor="0.75", legend=True, ax=ax_map)

    if scheme == "UserDefined":
        gdf.plot(**base, scheme="UserDefined", classification_kwds={"bins": bins})
    elif scheme == "StdMean":
        gdf.plot(**base, scheme="StdMean")
    else:
        gdf.plot(**base, scheme=scheme, k=k)

    ax_map.set_title(title or scheme)
    ax_map.axis("off")
    leg = ax_map.get_legend()
    if leg:
        leg.set_bbox_to_anchor((1.05, 1))

    plt.tight_layout()
    plt.show()

    print(f"Границы классов: {classifier.bins.round(1)}")
    print(f"Количество объектов в классах: {np.bincount(classifier.yb)}")


### 2.1. Equal Interval — равные интервалы

Диапазон значений (max − min) делится на `k` равных частей.

Метод интуитивно понятен и хорошо работает, когда данные распределены равномерно. Но он чувствителен к выбросам: одно экстремальное значение «растянет» шкалу, и большинство объектов окажутся в одном-двух классах.


In [ ]:
plot_classification(spb, "pop_density", "EqualInterval", k=5,
                    title="Equal Interval — равные интервалы")

### 2.2. Quantiles — квантили

Данные делятся так, чтобы в каждом классе было **одинаковое количество объектов**.

Все цвета на карте используются равномерно, карта выглядит насыщенной, и этот метод устойчив к выбросам.

Но объекты с очень похожими значениями могут попасть в разные классы, объекты с очень разными значениями — в один класс. Это может создавать визуально «ложные» различия.


In [ ]:
plot_classification(spb, "pop_density", "Quantiles", k=5,
                    title="Quantiles — квантили")

### 2.3. Natural Breaks (Jenks) — естественные границы

Алгоритм ищет границы там, где в данных есть **естественные разрывы**: минимизирует дисперсию внутри классов и максимизирует различия между ними. Такие классы отражают реальную структуру данных — это один из самых честных методов для большинства задач.

Из минусов: границы классов бывает сложно объяснить в легенде, и метод не подходит для сравнения карт за разные периоды — границы каждый раз будут разными. Но как универсальный вариант для социально-экономических показателей работает отлично.


In [ ]:
plot_classification(spb, "pop_density", "NaturalBreaks", k=5,
                    title="Natural Breaks (Jenks) — естественные границы")

### 2.4. Standard Deviation — стандартное отклонение

Классы строятся относительно среднего значения с шагом в одно стандартное отклонение. На карте сразу видно, насколько каждый округ отклоняется от нормы: «выше среднего», «ниже среднего», «намного выше».

Метод хорошо работает при нормально распределённых данных — тогда классы получаются осмысленными и легко читаются. Но при асимметричном распределении или выбросах (как в нашем случае) он теряет наглядность: количество классов определяется данными, а не параметром `k`, и часть диапазона может оказаться вообще не представленной.


In [ ]:
std = mc.StdMean(spb["pop_density"])
print(f"Среднее: {spb['pop_density'].mean():.0f} чел/км²")
print(f"Стандартное отклонение: {spb['pop_density'].std():.0f} чел/км²")
print(f"Границы классов: {std.bins.round(1)}")

plot_classification(spb, "pop_density", "StdMean",
                    title="Standard Deviation")

### 2.5. Maximum Breaks — максимальные разрывы

Находит `k−1` наибольших разрывов между соседними значениями (после сортировки) и использует их как границы классов. Границы ставятся именно там, где данные «прыгают» — метод хорошо работает, когда в данных есть явные кластеры.

Из минусов: чувствителен к выбросам — один большой разрыв у выброса может «съесть» один из классов. Плюс он менее распространён: в GeoPandas `.plot()` нет встроенной поддержки, классификатор придётся применять вручную.


In [ ]:
mb = mc.MaximumBreaks(spb["pop_density"], k=5)
print(f"Границы классов: {mb.bins.round(1)}")

plot_classification(spb, "pop_density", "MaximumBreaks", k=5,
                    title="Maximum Breaks — максимальные разрывы")

### 2.6. User Defined — пользовательские интервалы

Вы задаёте границы классов вручную. Конечно, это не отдельный статистический метод:) Есть два случая, когда мы его используем:

1. **Круглые числа.** Вы взяли Natural Breaks как основу, но подправили границы до «красивых» значений для легенды.
2. **Сравнение карт.** При сравнении двух временных срезов нужно использовать **одинаковые границы** на обеих картах — иначе сравнение некорректно.


In [ ]:
# Смотрим на Natural Breaks как отправную точку
nb = mc.NaturalBreaks(spb["pop_density"], k=5)
print("Natural Breaks (исходные границы):", nb.bins.round(0))

# Округляем до «читаемых» значений
custom_bins = [4000, 10000, 15000, 22000, 35000]

plot_classification(spb, "pop_density", scheme="UserDefined",
                    bins=custom_bins,
                    title="User Defined — пользовательские интервалы")


## 3. Сравнение всех методов

Посмотрим на все методы на одном графике — как каждый округ попадает в разные классы при разных подходах.


In [ ]:
# Применяем все методы и сохраняем классы в датафрейм
k = 5
classifiers = {
    "EqualInterval": mc.EqualInterval(spb["pop_density"], k=k),
    "Quantiles":     mc.Quantiles(spb["pop_density"], k=k),
    "NaturalBreaks": mc.NaturalBreaks(spb["pop_density"], k=k),
    "MaximumBreaks": mc.MaximumBreaks(spb["pop_density"], k=k),
    "StdMean":       mc.StdMean(spb["pop_density"]),
}

for name, clf in classifiers.items():
    spb[name] = clf.yb

In [ ]:
# Тепловая карта: строки = методы, столбцы = округа (отсортированы по плотности)
fig, ax = plt.subplots(figsize=(14, 4))

matrix = (spb.set_index("NAME")
             .sort_values("pop_density")[list(classifiers.keys())]
             .T)

ax.imshow(matrix.values, cmap="BuPu", aspect="auto")

ax.set_xticks(range(len(matrix.columns)))
ax.set_xticklabels(matrix.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(matrix.index)))
ax.set_yticklabels(matrix.index)

ax.set_xlabel("Округа (отсортированы по плотности населения →)")
ax.set_title("Сравнение методов: в какой класс попадает каждый округ")
plt.tight_layout()
plt.show()


## Итог

Метод классификации — это ваше аналитическое решение, которое влияет на то, что читатель увидит на карте. Одни и те же данные — разные карты.

- **Equal Interval** — простой и понятный, но при скошенном распределении большинство объектов окажутся в одном классе. Плотность населения Петербурга — как раз такой случай: несколько плотно заселённых округов «растягивают» шкалу.

- **Quantiles** — карта получается насыщенной, все цвета задействованы равномерно. Но метод может уравнивать объекты с очень разными значениями и разделять очень похожие.

- **Natural Breaks (Jenks)** — ищет границы там, где в данных есть реальные разрывы. Хороший выбор по умолчанию для большинства социально-экономических показателей.

- **Standard Deviation** — показывает не абсолютные значения, а отклонение от среднего. Удобно, когда важно показать «выше нормы» / «ниже нормы».

- **Maximum Breaks** — выделяет самые резкие скачки в данных.

- **User Defined** — мы часто "вручную" определяем классы на основе одной из существующих классификаций.
